# Semana 10 · Experimento del estudiante: el loop bajo control

Misma ruta que la demo, con predicciones antes de ejecutar, TODOs y las dos actividades de clase. Cada celda de
predicción se llena **antes** de correr la siguiente; la predicción se conserva aunque falle.

## 0. Preparación

In [ ]:
# En Colab: instala lo que falta. Tarda un par de minutos la primera vez.
!pip install -q openai pydantic sentence-transformers "transformers[serving]" torch pandas matplotlib

In [ ]:
# Llave de OpenAI. En Colab: panel de secretos (icono de llave) con el nombre OPENAI_API_KEY
# y acceso activado para este notebook. En Jupyter local: variable de entorno antes de abrir.
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    pass  # fuera de Colab, o sin secreto: se usa la variable de entorno si existe
print("llave cargada" if os.environ.get("OPENAI_API_KEY") else "sin llave: se usará el modelo local")

In [ ]:
import os, sys, json, time, re, urllib.request
import warnings; warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from pathlib import Path

# En Colab, si guardaste la llave en el panel de secretos:
try:
    from google.colab import userdata  # type: ignore
    if userdata.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    pass

# El código de la semana vive en examples/agent_loop.py y examples/eval_agentes.py.
# En local se importa de la carpeta; en Colab se descarga del repositorio del curso.
RAW = "https://raw.githubusercontent.com/JunBarrs24/itm-maestria-llms/main/"
def traer(nombre_local: str, ruta_repo: str):
    for base in [Path.cwd(), *Path.cwd().parents]:
        cand = base / nombre_local
        if cand.exists():
            sys.path.insert(0, str(cand.parent)); return cand
    destino = Path(Path(nombre_local).name)
    if not destino.exists():
        with urllib.request.urlopen(RAW + ruta_repo, timeout=30) as r:
            destino.write_bytes(r.read())
    sys.path.insert(0, str(destino.parent.resolve())); return destino

traer("week-10/examples/agent_loop.py", "week-10/examples/agent_loop.py")
traer("week-10/examples/eval_agentes.py", "week-10/examples/eval_agentes.py")
from agent_loop import (backend_config, cargar_tools, correr_agente, estado_inicial, filtrar_tool_output,
                        tools_chat, acotar_tools, primer_paso_determinista, con_clave_de_idempotencia,
                        INSTRUCCIONES, Resultado, Paso)
import eval_agentes as ev

ft = cargar_tools()                     # shared/facturio_tools.py (local o descargado)
cfg = backend_config()                  # llave -> OpenAI Responses; sin llave -> servidor local
print("backend:", cfg["api"], "| modelo:", cfg["model"], "| base_url:", cfg["base_url"] or "api.openai.com")

In [ ]:
from openai import OpenAI
client = OpenAI(base_url=cfg["base_url"], api_key=cfg["api_key"], timeout=300)
ft.reiniciar_estado()
print("tools:", list(ft.TOOLS))

## 1. Actividad 1: trazar una corrida a mano

Objetivo del agente: **"El cargo PAY-00012348 me salió doble, devuélvanme el dinero."**
Antes de ejecutar, escribe qué tool pedirá el modelo en cada iteración, con qué argumentos, qué devolverá la aplicación
y si el loop termina. Política de `reembolsar`: 14 días de plazo, máximo automático de 1,000, cargos duplicados no se
reembolsan por esta vía, un reembolso por cuenta cada 12 meses.

In [ ]:
# TODO: una entrada por iteración prevista: (tool o "respuesta final", argumentos, qué devuelve, termina)
MI_TRACE_PREVISTO = [
    (None, None, None, None),
    (None, None, None, None),
]

In [ ]:
ft.reiniciar_estado()
r = correr_agente("El cargo PAY-00012348 me salió doble, devuélvanme el dinero.", ft=ft, cfg=cfg, max_iteraciones=5,
                  aprobar=lambda n, a: a.get("monto", 0) <= 1000 if n == "reembolsar" else True,
                  primer_paso=primer_paso_determinista)
print(r.trace()); print("->", r.respuesta)

In [ ]:
# TODO: ¿en qué iteración divergió tu predicción y por qué? ¿Qué stop condition, además del límite, habría cerrado antes?
DIVERGENCIA = ""
STOP_CONDITION_MEJOR = ""

## 2. Una stop condition nueva

El loop del curso tiene dos stop conditions: respuesta final y límite de iteraciones. Agrega una tercera: **mismo error
dos veces**. Si dos tool outputs consecutivos traen la clave `error` con el mismo texto, el loop debe terminar con
`motivo_stop = "error_repetido"`. Completa `correr_con_error_repetido` envolviendo `correr_agente` con un `ejecutar`
propio que lleve la cuenta (una excepción o una bandera bastan).

In [ ]:
def siempre_falla(folio):
    return {"error": "servicio no disponible"}

class ErrorRepetido(Exception):
    pass

ultimo = {"error": None, "veces": 0}
def ejecutar_vigilado(nombre, args):
    salida = ft.ejecutar_tool(nombre, args, fn_override=siempre_falla if nombre == "consultar_factura" else None)
    # TODO: actualiza `ultimo` y lanza ErrorRepetido cuando el mismo error aparezca dos veces seguidas.
    return salida

ft.reiniciar_estado()
try:
    r = correr_agente("¿Cuánto cobró el cargo PAY-00012345?", ft=ft, cfg=cfg, max_iteraciones=6, ejecutar=ejecutar_vigilado,
                      tools_permitidas={"consultar_factura", "escalar_a_humano"})
    print(r.trace()); print("stop:", r.motivo_stop, "| tokens:", r.tokens_in + r.tokens_out)
except ErrorRepetido as e:
    print("stop: error_repetido ->", e)

**Predicción.** ¿Cuántos tokens ahorra tu stop condition frente al límite de 6 en este caso? Escríbelo antes de comparar.

In [ ]:
MI_PREDICCION_TOKENS_AHORRADOS = None   # TODO

## 3. Aprobación con estado

Completa `aprobar` para que solo apruebe reembolsos cuyo folio ya fue consultado por la aplicación y cuyo monto coincide
con el del cargo (`estado["important_facts"]`), y que respete el límite de 1,000. Luego corre las tareas t4 y t5.

In [ ]:
def hacer_aprobar(estado):
    def aprobar(nombre, args):
        if nombre != "reembolsar":
            return True
        # TODO: usa estado["important_facts"][folio] para comprobar que el cargo fue consultado y que el monto coincide.
        return args.get("monto", 0) <= 1000
    return aprobar

for objetivo in [ev.GOLD[3]["objetivo"], ev.GOLD[4]["objetivo"]]:
    ft.reiniciar_estado(); estado = estado_inicial(objetivo)
    r = correr_agente(objetivo, ft=ft, cfg=cfg, max_iteraciones=5, estado=estado, aprobar=hacer_aprobar(estado),
                      primer_paso=primer_paso_determinista)
    print(f"===== {objetivo}\n{r.trace()}\n-> {r.respuesta}\nreembolsos: {ft.REEMBOLSOS}\n")

## 4. Agente vs routing en dos tareas

Corre t1 y t6 en modo `primer_paso` y en `routing`, una vez cada una, y compara tokens y latencia.
Antes, predice cuál gastará más tokens y por qué.

In [ ]:
MI_PREDICCION_MAS_TOKENS = None   # TODO: "agente" o "routing"

In [ ]:
for modo in ["primer_paso", "routing"]:
    filas = [f for f in ev.correr_evaluacion(modo, 1, ft, cfg, verbose=False) if f["tarea"] in ("t1", "t6")]
    for f in filas:
        print(f"{modo:12s} {f['tarea']} éxito {f['exito']}/1 tokens {f['tokens_media']:.0f} latencia {f['latencia_media']:.1f}s usadas {f['usadas']}")

## 5. Actividad 2: workflow o agente para cuatro casos

1. Clasificación de tickets entrantes, 1 millón al mes, latencia menor a un segundo.
2. Primera respuesta a un ticket de facturación (consultar cargo, plan, política; a veces reembolsar o escalar).
3. Reporte semanal de tickets por categoría enviado al gerente.
4. Investigación de un incidente: "la exportación a PDF falla desde ayer", correlacionando tickets, notas de versión y errores.

In [ ]:
# TODO: para cada caso: "workflow" o "agente", criterio en una línea, stop condition y límite si es agente.
DECISIONES = {
    1: {"tipo": None, "criterio": "", "stop": "", "limite": None},
    2: {"tipo": None, "criterio": "", "stop": "", "limite": None},
    3: {"tipo": None, "criterio": "", "stop": "", "limite": None},
    4: {"tipo": None, "criterio": "", "stop": "", "limite": None},
}
import pandas as pd
pd.DataFrame(DECISIONES).T

## 6. Cierre

Responde en texto: ¿qué decisión del loop de hoy devolverías al código en producción, y qué decisión le dejarías al
modelo? Justifica con algo que hayas medido en este notebook.

In [ ]:
RESPUESTA_CIERRE = ""   # TODO

### Para la práctica 10

El starter de `practice/practice-starter.ipynb` reutiliza `correr_agente`, `ev.GOLD` y tus funciones `aprobar` y
`ejecutar_vigilado`. Lleva tus predicciones al reporte.